# Extract SIF activations for the NARCBench probe (gpt-oss-20b)

Runs on **T4 x2** (set via the API at push time). Downloads gpt-oss-20b, loads it
across both T4s with the ~12 GB overflow offloaded to CPU RAM, teacher-forces all
123 turns, and saves one layer-12 activation vector each to `sif_acts.npz`.

The model load is done here (with `max_memory` so it fits) but the extraction itself
reuses the defense's own `extract_activations.extract` unchanged — same hook, same
`build_gen_text`, so the vectors match what the probe was trained on.

In [ ]:
import os, glob, subprocess, sys

# locate the 4 uploaded files anywhere under /kaggle/input
NEEDED = ["score_sif_vs_narcbench.py", "extract_activations.py",
          "sif_turns.json", "narcbench_probe_gpt_oss_20b.pkl"]
found = {}
for name in NEEDED:
    hits = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    if not hits:
        raise FileNotFoundError(f"'{name}' not found under /kaggle/input -- attach the dataset (Add Input).")
    found[name] = hits[0]
    print("found:", hits[0])

# gpt-oss needs a recent transformers; installed into the kernel env before we import it below
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U",
                "transformers>=4.55", "accelerate"], check=True)

smi = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout
print("\nGPUs:\n" + smi)

In [ ]:
# Load in-kernel with max_memory so the ~42 GB bf16 model fits on 2x T4 (~30 GB) + CPU
# offload, then hand the model to the defense's own extract() (unchanged hook/logic).
import os, sys, json, pickle
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"   # cut fragmentation
import numpy as np, torch

sys.path.insert(0, os.path.dirname(found["extract_activations.py"]))
import extract_activations as ea          # brings torch/transformers + the GELU shim
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "openai/gpt-oss-20b"
layer = pickle.load(open(found["narcbench_probe_gpt_oss_20b.pkl"], "rb"))["layer"]
turns = json.load(open(found["sif_turns.json"]))
print(f"loading {model_name} (layer {layer}) across", torch.cuda.device_count(), "GPUs ...")

os.makedirs("/kaggle/tmp/offload", exist_ok=True)
tok = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    model_name, trust_remote_code=True, device_map="auto",
    max_memory={0: "13GiB", 1: "13GiB", "cpu": "26GiB"},   # leave GPU headroom -> spill to CPU, not OOM
    offload_folder="/kaggle/tmp/offload")
model.eval()
print("device map (sample):", dict(list(getattr(model, "hf_device_map", {}).items())[:4]), "...")

acts = ea.extract(model, tok, turns, [layer])              # {layer: (n_turns, hidden)}
np.savez_compressed("/kaggle/working/sif_acts.npz", **{f"layer_{layer}": acts[layer]})
print(f"saved {acts[layer].shape} -> /kaggle/working/sif_acts.npz")

In [ ]:
import numpy as np, json
acts = np.load("/kaggle/working/sif_acts.npz")
key = acts.files[0]
n_turns = len(json.load(open(found["sif_turns.json"])))
print("saved:", key, acts[key].shape)
assert acts[key].shape[0] == n_turns, f"row mismatch: {acts[key].shape[0]} vs {n_turns} turns"
print(f"OK: {acts[key].shape[0]} vectors x dim {acts[key].shape[1]}  (expect 123 x 2880)")
print("\nsif_acts.npz is in the Output panel.")